# 📈 Stock Price Prediction — Advanced LSTM with Attention & Technical Indicators

**Architecture Highlights:**
- Multi-feature LSTM with engineered financial indicators (RSI, MACD, Bollinger Bands)
- Custom Attention mechanism layer
- Buy / Sell / Hold signal generation
- Backtesting with cumulative returns vs. Buy-and-Hold benchmark
- Interactive Plotly visualizations
- Experiment tracking with MLflow

> **Stack:** Python · TensorFlow/Keras · yfinance · TA-Lib · Plotly · MLflow · scikit-learn

## 1. Install & Import Dependencies

In [1]:
# Install required libraries
!pip install yfinance ta plotly mlflow scikit-learn tensorflow --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import yfinance as yf
import ta  # Technical Analysis library

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow
import mlflow.keras
import warnings
import os

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.6/887.6 kB 27.8 MB/s eta 0:00:00
   ━━━

## 2. Configuration

In [2]:
# ── Easily configurable parameters ──────────────────────────────────────────
CONFIG = {
    'ticker'      : 'AAPL',      # Stock ticker — change to any symbol
    'start_date'  : '2018-01-01',
    'end_date'    : '2024-12-31',
    'sequence_len': 60,          # Lookback window (trading days)
    'test_split'  : 0.15,        # 15% test set
    'val_split'   : 0.15,        # 15% validation set
    'epochs'      : 100,
    'batch_size'  : 32,
    'learning_rate': 1e-3,
    'lstm_units'  : [128, 64, 32],  # Stacked LSTM sizes
    'dropout_rate': 0.3,
    'target_col'  : 'Close',
}

print('Configuration set:', CONFIG)

Configuration set: {'ticker': 'AAPL', 'start_date': '2018-01-01', 'end_date': '2024-12-31', 'sequence_len': 60, 'test_split': 0.15, 'val_split': 0.15, 'epochs': 100, 'batch_size': 32, 'learning_rate': 0.001, 'lstm_units': [128, 64, 32], 'dropout_rate': 0.3, 'target_col': 'Close'}


## 3. Data Collection & Feature Engineering

In [3]:
def download_and_engineer_features(ticker, start, end):
    """Download OHLCV data and engineer technical indicators."""
    print(f'Downloading {ticker} data...')
    df = yf.download(ticker, start=start, end=end, progress=False)
    df.columns = df.columns.get_level_values(0)  # Flatten MultiIndex if present
    df.dropna(inplace=True)

    # ── Price-based features ────────────────────────────────────────────────
    df['Returns']       = df['Close'].pct_change()
    df['Log_Returns']   = np.log(df['Close'] / df['Close'].shift(1))
    df['HL_Ratio']      = (df['High'] - df['Low']) / df['Close']   # Daily range
    df['OC_Ratio']      = (df['Close'] - df['Open']) / df['Open']  # Body size

    # ── Moving Averages ─────────────────────────────────────────────────────
    df['SMA_10']  = ta.trend.sma_indicator(df['Close'], window=10)
    df['SMA_30']  = ta.trend.sma_indicator(df['Close'], window=30)
    df['EMA_12']  = ta.trend.ema_indicator(df['Close'], window=12)
    df['EMA_26']  = ta.trend.ema_indicator(df['Close'], window=26)
    df['Price_SMA10_Ratio'] = df['Close'] / df['SMA_10']

    # ── Momentum Indicators ──────────────────────────────────────────────────
    df['RSI']     = ta.momentum.rsi(df['Close'], window=14)
    macd          = ta.trend.MACD(df['Close'])
    df['MACD']    = macd.macd()
    df['MACD_Signal'] = macd.macd_signal()
    df['MACD_Diff']   = macd.macd_diff()

    # ── Volatility Indicators ────────────────────────────────────────────────
    bb            = ta.volatility.BollingerBands(df['Close'], window=20)
    df['BB_High'] = bb.bollinger_hband()
    df['BB_Low']  = bb.bollinger_lband()
    df['BB_Width']= (df['BB_High'] - df['BB_Low']) / df['Close']
    df['BB_Pos']  = (df['Close'] - df['BB_Low']) / (df['BB_High'] - df['BB_Low'] + 1e-8)
    df['ATR']     = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'])

    # ── Volume Indicators ────────────────────────────────────────────────────
    df['Volume_SMA'] = ta.trend.sma_indicator(df['Volume'], window=20)
    df['Volume_Ratio']  = df['Volume'] / (df['Volume_SMA'] + 1)
    df['OBV']        = ta.volume.on_balance_volume(df['Close'], df['Volume'])

    df.dropna(inplace=True)
    print(f'Dataset shape after feature engineering: {df.shape}')
    print(f'Features: {list(df.columns)}')
    return df


df = download_and_engineer_features(
    CONFIG['ticker'], CONFIG['start_date'], CONFIG['end_date']
)

Dataset shape after feature engineering: (1727, 26)
Features: ['Close', 'High', 'Low', 'Open', 'Volume', 'Returns', 'Log_Returns', 'HL_Ratio', 'OC_Ratio', 'SMA_10', 'SMA_30', 'EMA_12', 'EMA_26', 'Price_SMA10_Ratio', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Diff', 'BB_High', 'BB_Low', 'BB_Width', 'BB_Pos', 'ATR', 'Volume_SMA', 'Volume_Ratio', 'OBV']


## 4. Exploratory Data Analysis (EDA)

In [4]:
def plot_eda(df, ticker):
    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        subplot_titles=(
            f'{ticker} Close Price with Bollinger Bands & MAs',
            'RSI (14)',
            'MACD',
            'Volume'
        ),
        row_heights=[0.45, 0.18, 0.18, 0.19]
    )

    # Candlestick
    fig.add_trace(go.Candlestick(
        x=df.index, open=df['Open'], high=df['High'],
        low=df['Low'], close=df['Close'], name='OHLC'
    ), row=1, col=1)

    for col, color, name in [
        ('SMA_10','#FFA500','SMA 10'), ('SMA_30','#0080FF','SMA 30'),
        ('BB_High','#00CC00','BB High'), ('BB_Low','#CC0000','BB Low')
    ]:
        fig.add_trace(go.Scatter(
            x=df.index, y=df[col], line=dict(color=color, width=1),
            name=name, opacity=0.8
        ), row=1, col=1)

    # RSI
    fig.add_trace(go.Scatter(x=df.index, y=df['RSI'], line=dict(color='purple', width=1.5), name='RSI'), row=2, col=1)
    for level, color in [(70, 'red'), (30, 'green')]:
        fig.add_hline(y=level, line_dash='dash', line_color=color, row=2, col=1)

    # MACD
    fig.add_trace(go.Scatter(x=df.index, y=df['MACD'], line=dict(color='blue', width=1), name='MACD'), row=3, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['MACD_Signal'], line=dict(color='orange', width=1), name='Signal'), row=3, col=1)
    fig.add_trace(go.Bar(x=df.index, y=df['MACD_Diff'], name='MACD Diff', marker_color='gray', opacity=0.5), row=3, col=1)

    # Volume
    fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='Volume', marker_color='steelblue', opacity=0.6), row=4, col=1)

    fig.update_layout(height=900, title_text=f'{ticker} — Technical Analysis Dashboard', showlegend=True, xaxis_rangeslider_visible=False)
    fig.show()


plot_eda(df, CONFIG['ticker'])

## 5. Data Preprocessing & Sequence Creation

In [5]:
# Select feature columns (exclude raw OHLCV that are already captured by indicators)
FEATURE_COLS = [
    'Close', 'Volume',
    'Returns', 'Log_Returns', 'HL_Ratio', 'OC_Ratio',
    'SMA_10', 'SMA_30', 'EMA_12', 'EMA_26', 'Price_SMA10_Ratio',
    'RSI', 'MACD', 'MACD_Signal', 'MACD_Diff',
    'BB_Width', 'BB_Pos', 'ATR',
    'Volume_Ratio', 'OBV'
]

TARGET_COL_IDX = FEATURE_COLS.index('Close')  # Index of Close in feature array

data = df[FEATURE_COLS].values
print(f'Total features: {len(FEATURE_COLS)}')
print(f'Total samples : {len(data)}')

# ── Train / Val / Test split (chronological) ────────────────────────────────
n = len(data)
test_size = int(n * CONFIG['test_split'])
val_size  = int(n * CONFIG['val_split'])
train_size = n - val_size - test_size

train_data = data[:train_size]
val_data   = data[train_size:train_size + val_size]
test_data  = data[train_size + val_size:]

# ── Scaling (fit only on train) ─────────────────────────────────────────────
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_data)
val_scaled   = scaler.transform(val_data)
test_scaled  = scaler.transform(test_data)

# Separate scaler for inverse-transforming Close price
close_scaler = MinMaxScaler(feature_range=(0, 1))
close_scaler.fit(train_data[:, [TARGET_COL_IDX]])


def create_sequences(data, seq_len, target_idx):
    X, y = [], []
    for i in range(seq_len, len(data)):
        X.append(data[i - seq_len:i])      # All features
        y.append(data[i, target_idx])       # Next Close price
    return np.array(X), np.array(y)


SEQ = CONFIG['sequence_len']
X_train, y_train = create_sequences(train_scaled, SEQ, TARGET_COL_IDX)
X_val,   y_val   = create_sequences(val_scaled,   SEQ, TARGET_COL_IDX)
X_test,  y_test  = create_sequences(test_scaled,  SEQ, TARGET_COL_IDX)

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val  : {X_val.shape}    y_val  : {y_val.shape}')
print(f'X_test : {X_test.shape}   y_test : {y_test.shape}')

Total features: 20
Total samples : 1727
X_train: (1149, 60, 20)  y_train: (1149,)
X_val  : (199, 60, 20)    y_val  : (199,)
X_test : (199, 60, 20)   y_test : (199,)


## 6. Model Architecture — LSTM with Attention

In [6]:
class AttentionLayer(layers.Layer):
    """Bahdanau-style additive attention over LSTM output sequences."""

    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units, use_bias=False)
        self.V = layers.Dense(1, use_bias=False)

    def call(self, encoder_output):
        # encoder_output: (batch, seq_len, hidden_size)
        score       = self.V(tf.nn.tanh(self.W(encoder_output)))  # (batch, seq_len, 1)
        weights     = tf.nn.softmax(score, axis=1)                 # (batch, seq_len, 1)
        context     = weights * encoder_output                     # (batch, seq_len, hidden)
        context_sum = tf.reduce_sum(context, axis=1)              # (batch, hidden)
        return context_sum, tf.squeeze(weights, -1)                # also return weights for viz

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.W.units})
        return config


def build_lstm_attention_model(seq_len, n_features, lstm_units, dropout_rate, lr):
    inputs = keras.Input(shape=(seq_len, n_features), name='input')
    x = inputs

    # Stacked LSTM layers — all but last return sequences
    for i, units in enumerate(lstm_units):
        return_seq = True  # Always True so Attention can see all timesteps
        x = layers.LSTM(
            units,
            return_sequences=return_seq,
            dropout=dropout_rate,
            recurrent_dropout=0.0,
            name=f'lstm_{i}'
        )(x)
        if i < len(lstm_units) - 1:
            x = layers.LayerNormalization(name=f'ln_{i}')(x)

    # Attention
    context, attention_weights = AttentionLayer(units=64, name='attention')(x)

    # Dense head
    x = layers.Dense(64, activation='relu', name='dense_1')(context)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(32, activation='relu', name='dense_2')(x)
    output = layers.Dense(1, name='output')(x)

    model = Model(inputs=inputs, outputs=output, name='LSTM_Attention_StockPredictor')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='huber',   # Huber loss — more robust to outliers than MSE
        metrics=['mae']
    )
    return model


model = build_lstm_attention_model(
    seq_len     = SEQ,
    n_features  = len(FEATURE_COLS),
    lstm_units  = CONFIG['lstm_units'],
    dropout_rate= CONFIG['dropout_rate'],
    lr          = CONFIG['learning_rate']
)
model.summary()

Model: "LSTM_Attention_StockPredictor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 60, 20)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_0 (LSTM)                   │ (None, 60, 128)        │        76,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ln_0 (LayerNormalization)       │ (None, 60, 128)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 60, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ln_1 (LayerNormalization)       │ (None, 60, 64)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 60, 32)         │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention (AttentionLayer)      │ [(None, 32), (None,    │         2,112 │
│                                 │ 60)]                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 144,833 (565.75 KB)

 Trainable params: 144,833 (565.75 KB)

 Non-trainable params: 0 (0.00 B)

## 7. Training with Callbacks & MLflow Logging

In [7]:
os.makedirs('models', exist_ok=True)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    ModelCheckpoint('models/best_model.keras', monitor='val_loss', save_best_only=True, verbose=0),
]

mlflow.set_experiment('stock_lstm_attention')

with mlflow.start_run(run_name=f"{CONFIG['ticker']}_attention_lstm"):
    mlflow.log_params({
        'ticker'       : CONFIG['ticker'],
        'lstm_units'   : str(CONFIG['lstm_units']),
        'dropout_rate' : CONFIG['dropout_rate'],
        'sequence_len' : CONFIG['sequence_len'],
        'n_features'   : len(FEATURE_COLS),
        'learning_rate': CONFIG['learning_rate'],
        'batch_size'   : CONFIG['batch_size'],
    })

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs     = CONFIG['epochs'],
        batch_size = CONFIG['batch_size'],
        callbacks  = callbacks,
        verbose    = 1
    )

    # Log final metrics
    best_val_loss = min(history.history['val_loss'])
    best_val_mae  = min(history.history['val_mae'])
    mlflow.log_metrics({'best_val_loss': best_val_loss, 'best_val_mae': best_val_mae})
    mlflow.keras.log_model(model, 'model')

print(f'Best val_loss: {best_val_loss:.6f}  |  Best val_mae: {best_val_mae:.6f}')

2026/05/17 21:30:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/17 21:30:20 INFO mlflow.store.db.utils: Updating database tables
2026/05/17 21:30:22 INFO mlflow.tracking.fluent: Experiment with name 'stock_lstm_attention' does not exist. Creating a new experiment.


Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - loss: 0.0593 - mae: 0.2679 - val_loss: 0.0097 - val_mae: 0.1242 - learning_rate: 0.0010
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0197 - mae: 0.1544 - val_loss: 0.0487 - val_mae: 0.3029 - learning_rate: 0.0010
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0146 - mae: 0.1244 - val_loss: 0.0409 - val_mae: 0.2762 - learning_rate: 0.0010
Epoch 4/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0111 - mae: 0.1135 - val_loss: 0.0433 - val_mae: 0.2841 - learning_rate: 0.0010
Epoch 5/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0112 - mae: 0.1123 - val_loss: 0.0400 - val_mae: 0.2723 - learning_rate: 0.0010
Epoch 6/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0110 - mae: 0.1101 - val_loss: 0.0344 - val_mae: 0.2510 - learning_rate: 0.0010
Epoch 7/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0095 - mae: 0.1029 - val_loss: 0.0554 - val_mae: 0.3239 - learning_rate: 0.0010
Epoch 

2026/05/17 21:30:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 21:30:43 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2026/05/17 21:30:57 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpg8td245w/model, flavor: keras). Fall back to return ['keras==3.13.2']. Set logging level to DEBUG to see the full traceback. 


Best val_loss: 0.009746  |  Best val_mae: 0.124158


## 8. Training History Plot

In [8]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss (Huber)', 'MAE'))
for metric, col in [('loss', 1), ('mae', 2)]:
    fig.add_trace(go.Scatter(y=history.history[metric],        name='Train', line=dict(color='steelblue')), row=1, col=col)
    fig.add_trace(go.Scatter(y=history.history[f'val_{metric}'], name='Val',   line=dict(color='salmon')),    row=1, col=col)
fig.update_layout(height=400, title='Training History', showlegend=True)
fig.show()

## 9. Evaluation & Predictions

In [9]:
y_pred_scaled = model.predict(X_test, verbose=0)

# Inverse-transform to original price scale
y_pred_price = close_scaler.inverse_transform(y_pred_scaled)
y_true_price = close_scaler.inverse_transform(y_test.reshape(-1, 1))

mae   = mean_absolute_error(y_true_price, y_pred_price)
rmse  = np.sqrt(mean_squared_error(y_true_price, y_pred_price))
mape  = np.mean(np.abs((y_true_price - y_pred_price) / (y_true_price + 1e-8))) * 100
r2    = r2_score(y_true_price, y_pred_price)

print('=' * 45)
print(f'  Test Set Performance — {CONFIG["ticker"]}')
print('=' * 45)
print(f'  MAE  : ${mae:.2f}')
print(f'  RMSE : ${rmse:.2f}')
print(f'  MAPE : {mape:.2f}%')
print(f'  R²   : {r2:.4f}')
print('=' * 45)

with mlflow.start_run(run_name=f"{CONFIG['ticker']}_eval", nested=True):
    mlflow.log_metrics({'test_mae': mae, 'test_rmse': rmse, 'test_mape': mape, 'test_r2': r2})

  Test Set Performance — AAPL
  MAE  : $49.02
  RMSE : $54.77
  MAPE : 22.03%
  R²   : -3.8827


## 10. Prediction vs Actual Chart

In [10]:
# Align dates with test predictions
test_dates = df.index[train_size + val_size + SEQ:]

fig = go.Figure()
fig.add_trace(go.Scatter(x=test_dates, y=y_true_price.flatten(), name='Actual',    line=dict(color='steelblue', width=2)))
fig.add_trace(go.Scatter(x=test_dates, y=y_pred_price.flatten(), name='Predicted', line=dict(color='tomato',    width=2, dash='dot')))
fig.update_layout(
    title=f'{CONFIG["ticker"]} — Actual vs Predicted Close Price (Test Set)',
    xaxis_title='Date', yaxis_title='Price (USD)',
    height=450, hovermode='x unified'
)
fig.show()
print(f'MAPE: {mape:.2f}%  |  R²: {r2:.4f}')

MAPE: 22.03%  |  R²: -3.8827


## 11. Trading Signal Generator & Backtesting

In [11]:
def generate_signals(actual_prices, predicted_prices, threshold=0.005):
    """
    Generate Buy / Sell / Hold signals.
    threshold: minimum predicted % move to trigger a trade (filters noise).
    """
    signals = []
    for i in range(len(predicted_prices)):
        pct_change = (predicted_prices[i] - actual_prices[i]) / actual_prices[i]
        if pct_change > threshold:
            signals.append(1)    # BUY
        elif pct_change < -threshold:
            signals.append(-1)   # SELL
        else:
            signals.append(0)    # HOLD
    return np.array(signals)


def backtest(actual_prices, signals, initial_capital=100_000):
    """
    Simulate a simple long/short strategy.
    Position = 1 share when signal=BUY, -1 when SELL, 0 when HOLD.
    """
    portfolio = []
    cash = initial_capital
    position = 0          # Shares held
    shares = 0

    for i, (price, signal) in enumerate(zip(actual_prices, signals)):
        if signal == 1 and position == 0:    # BUY
            shares   = int(cash / price)
            cash    -= shares * price
            position = 1
        elif signal == -1 and position == 1: # SELL
            cash    += shares * price
            shares   = 0
            position = 0

        portfolio_value = cash + shares * price
        portfolio.append(portfolio_value)

    # Final liquidation
    if shares > 0:
        portfolio[-1] = cash + shares * actual_prices[-1]

    return np.array(portfolio)


actual   = y_true_price.flatten()
predicted= y_pred_price.flatten()

signals  = generate_signals(actual, predicted, threshold=0.005)
strategy_portfolio  = backtest(actual, signals)

# Benchmark: buy-and-hold
initial_capital = 100_000
shares_bh       = int(initial_capital / actual[0])
bh_portfolio    = actual * shares_bh + (initial_capital - shares_bh * actual[0])

# Returns
strategy_return = (strategy_portfolio[-1] - initial_capital) / initial_capital * 100
bh_return       = (bh_portfolio[-1]       - initial_capital) / initial_capital * 100

print(f'Initial Capital     : ${initial_capital:,.0f}')
print(f'Strategy Return     : {strategy_return:.2f}%  (${strategy_portfolio[-1]:,.0f})')
print(f'Buy & Hold Return   : {bh_return:.2f}%  (${bh_portfolio[-1]:,.0f})')
print(f'Signal Distribution : BUY={np.sum(signals==1)}, SELL={np.sum(signals==-1)}, HOLD={np.sum(signals==0)}')

Initial Capital     : $100,000
Strategy Return     : 0.00%  ($100,000)
Buy & Hold Return   : 45.67%  ($145,669)
Signal Distribution : BUY=0, SELL=198, HOLD=1


## 12. Backtest Visualization

In [12]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('Portfolio Value', 'Trading Signals'),
    row_heights=[0.6, 0.4]
)

# Portfolio comparison
fig.add_trace(go.Scatter(x=test_dates, y=strategy_portfolio, name='LSTM Strategy',
                         line=dict(color='green', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=test_dates, y=bh_portfolio,       name='Buy & Hold',
                         line=dict(color='gray',  width=2, dash='dot')), row=1, col=1)
fig.add_hline(y=initial_capital, line_dash='dash', line_color='black', opacity=0.4, row=1, col=1)

# Signals on price
fig.add_trace(go.Scatter(x=test_dates, y=actual, name='Price', line=dict(color='steelblue', width=1.5)), row=2, col=1)

buy_dates  = test_dates[signals == 1]
sell_dates = test_dates[signals == -1]
buy_prices  = actual[signals == 1]
sell_prices = actual[signals == -1]

fig.add_trace(go.Scatter(x=buy_dates,  y=buy_prices,  mode='markers', name='BUY',
                         marker=dict(color='green', symbol='triangle-up',   size=10)), row=2, col=1)
fig.add_trace(go.Scatter(x=sell_dates, y=sell_prices, mode='markers', name='SELL',
                         marker=dict(color='red',   symbol='triangle-down', size=10)), row=2, col=1)

fig.update_layout(
    height=650,
    title=f'{CONFIG["ticker"]} — Backtest: LSTM Strategy vs Buy & Hold  |  Strategy: {strategy_return:.1f}%  |  B&H: {bh_return:.1f}%',
    hovermode='x unified'
)
fig.show()

## 13. Attention Weights Visualization

In [14]:
# ── Attention Weights Visualization (Fixed) ──────────────────────────────────

# Build a sub-model that outputs ONLY the attention weights (second output)
lstm_output = model.get_layer('attention').input        # input going into attention
context, attn_weights_tensor = model.get_layer('attention')(lstm_output)

attention_model = Model(
    inputs=model.input,
    outputs=attn_weights_tensor   # only the weights tensor, not the tuple
)

# Get attention weights for a sample
sample_idx  = 0
sample_input = X_test[sample_idx:sample_idx + 1]
attn_weights = attention_model.predict(sample_input, verbose=0)  # shape: (1, seq_len)

# Plot
fig = go.Figure(go.Bar(
    x=list(range(SEQ)),
    y=attn_weights[0],
    marker=dict(
        color=attn_weights[0],
        colorscale='Viridis',
        showscale=True
    )
))
fig.update_layout(
    title='Attention Weights — Which Timesteps the Model Focuses On',
    xaxis_title='Timestep (0 = oldest, SEQ-1 = most recent)',
    yaxis_title='Attention Weight',
    height=380
)
fig.show()

## 14. Future Price Forecast (Next 30 Days)

In [15]:
def forecast_future(model, last_sequence, close_scaler, n_days=30):
    """
    Iteratively predict the next n_days Close prices.
    NOTE: Only the Close feature is updated during iteration;
    other features are kept constant from the last known window.
    """
    seq = last_sequence.copy()   # (seq_len, n_features)
    preds = []

    for _ in range(n_days):
        x = seq[np.newaxis, ...]   # (1, seq_len, n_features)
        pred_scaled = model.predict(x, verbose=0)[0, 0]
        preds.append(pred_scaled)

        # Shift window and insert prediction at the Close position
        new_row = seq[-1].copy()
        new_row[TARGET_COL_IDX] = pred_scaled
        seq = np.vstack([seq[1:], new_row])

    forecasted_prices = close_scaler.inverse_transform(
        np.array(preds).reshape(-1, 1)
    ).flatten()
    return forecasted_prices


last_seq = X_test[-1]   # Last sequence in test set
future_prices = forecast_future(model, last_seq, close_scaler, n_days=30)

import pandas as pd
last_date = df.index[-1]
future_dates = pd.bdate_range(start=last_date, periods=31)[1:]  # Business days

fig = go.Figure()
# Last 60 actual days for context
fig.add_trace(go.Scatter(
    x=df.index[-60:], y=df['Close'].values[-60:],
    name='Historical', line=dict(color='steelblue', width=2)
))
fig.add_trace(go.Scatter(
    x=future_dates, y=future_prices,
    name='30-Day Forecast',
    line=dict(color='orange', width=2, dash='dot'),
    mode='lines+markers', marker=dict(size=6)
))
fig.update_layout(
    title=f'{CONFIG["ticker"]} — 30-Day Price Forecast',
    xaxis_title='Date', yaxis_title='Price (USD)',
    height=400, hovermode='x unified'
)
fig.show()
print(f'Last known close : ${df["Close"].iloc[-1]:.2f}')
print(f'Forecast range   : ${future_prices.min():.2f} – ${future_prices.max():.2f}')

Last known close : $250.60
Forecast range   : $163.74 – $164.27


## 15. Save Artifacts for GitHub

In [16]:
import json

os.makedirs('artifacts', exist_ok=True)

# Save model
model.save('artifacts/lstm_attention_model.keras')

# Save scalers
import pickle
with open('artifacts/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('artifacts/close_scaler.pkl', 'wb') as f:
    pickle.dump(close_scaler, f)

# Save metrics to JSON
metrics = {
    'ticker'          : CONFIG['ticker'],
    'mae'             : round(float(mae), 4),
    'rmse'            : round(float(rmse), 4),
    'mape'            : round(float(mape), 4),
    'r2'              : round(float(r2), 4),
    'strategy_return' : round(float(strategy_return), 2),
    'bh_return'       : round(float(bh_return), 2),
    'n_features'      : len(FEATURE_COLS),
    'lstm_units'      : CONFIG['lstm_units'],
    'sequence_len'    : SEQ,
}
with open('artifacts/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved to artifacts/:')
print('  lstm_attention_model.keras')
print('  scaler.pkl')
print('  close_scaler.pkl')
print('  metrics.json')
print()
print(json.dumps(metrics, indent=2))

Saved to artifacts/:
  lstm_attention_model.keras
  scaler.pkl
  close_scaler.pkl
  metrics.json

{
  "ticker": "AAPL",
  "mae": 49.0157,
  "rmse": 54.7743,
  "mape": 22.0323,
  "r2": -3.8827,
  "strategy_return": 0.0,
  "bh_return": 45.67,
  "n_features": 20,
  "lstm_units": [
    128,
    64,
    32
  ],
  "sequence_len": 60
}
